# 🐱 Hidden Bias Steganography
### Embedding Secret Messages Inside a Neural Network's Bias Values

> *"What if a neural network was secretly whispering something to you this whole time?"*

---

**Author:** Pandora41  
**Status:** Ongoing Research  
**Paper Draft:** [Embedding Data inside Neural Network Bias Model: A Steganographic Approach Using Keras Models](https://docs.google.com/document/d/1FmyEre04vu4t17ZvqIxB2F5ku5cUlZJ6QiYmIGt1yvg/edit?usp=sharing)

---

## What Is This Notebook?

This notebook walks through the full pipeline of **neural network steganography using bias values** — from training a model, to hiding a secret message inside it, to recovering the message back.

Everything is in one place, explained step by step. nyaa~

### Table of Contents
1. [Background — What Is Steganography?](#background)
2. [How the Algorithm Works](#algorithm)
3. [Imports & Setup](#imports)
4. [Step 1 — Build & Train the Neural Network](#train)
5. [Step 2 — Encode the Secret Message into Bias Values](#encode)
6. [Step 3 — Save the Model](#save)
7. [Step 4 — Decode the Message from the Saved Model](#decode)
8. [Capacity Analysis](#capacity)
9. [Limitations & Future Work](#limitations)

---
<a id='background'></a>
## 1. Background — What Is Steganography?

**Steganography** is the art of hiding information inside something that looks completely normal.

Classic examples:
- Hiding text in the pixels of an image (the image looks the same to your eyes, but each pixel carries a hidden bit)
- Writing in invisible ink
- Encoding messages in the spacing between words in a document

This project brings steganography into the world of **machine learning** — specifically, hiding data inside the internal numbers of a neural network model.

### Neural Network Refresher

A neural network is made of **layers**. Each layer is made of **neurons**. Each neuron has two types of learned numbers:

| Parameter | What It Does | Typical Value After Training |
|---|---|---|
| **Weight** | Controls how strongly the neuron responds to each input | Small decimals (e.g., `0.023`, `-1.47`) |
| **Bias** | A personal offset — shifts the neuron's output up or down | Small decimals (e.g., `0.14`, `-0.03`) |

### Why Biases?

Bias values are perfect hiding spots because:
- Nobody usually inspects them directly
- They're stored as `float32` numbers, which can represent small integers (like ASCII codes 0–127) **exactly** with no rounding error
- There are enough of them across the layers to carry a useful message

Think of it like hiding a note inside a textbook — the textbook still works as a textbook, but someone who knows where to look will find your message. 🐾

---
<a id='algorithm'></a>
## 2. How the Algorithm Works

### Encoding (Hiding)

```
Secret message:  "Cats are cute"
                       ↓
   Convert each character to its ASCII integer
                       ↓
  C=67  a=97  t=116  s=115  ' '=32  a=97  r=114  e=101 ...
                       ↓
  Replace the first N bias values in the hidden layer
  with these ASCII integers (as float32)
                       ↓
  Save model → model_with_hidden_message.h5
```

### Decoding (Recovering)

```
  Load model from .h5 file
                       ↓
  Read the bias values from each Dense layer
                       ↓
  Round each float32 value to nearest integer
                       ↓
  Keep only values in ASCII range [0, 255]
                       ↓
  Convert each integer → character via chr()
                       ↓
  Read the secret! 🐾
```

### Why Does float32 Not Lose the Data?

The `float32` format can represent all integers from 0 to 16,777,216 **exactly** (no rounding). ASCII values only go up to 127, so our encoded values sit well within this safe range. When we save `67.0` into a bias and load it back, we get `67.0` — round it to int → `67` → `chr(67)` → `'C'`. Perfect, lossless recovery every time.

### Use Cases

| Use Case | How It Works |
|---|---|
| **CTF Challenge** | Give someone a `.h5` file. They must figure out the flag is in the bias values. |
| **Model Watermarking** | Embed your name/signature before releasing a model. Prove ownership if it gets stolen. |

---
<a id='imports'></a>
## 3. Imports & Setup

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import os

# Seed everything so results are reproducible
np.random.seed(42)
tf.random.set_seed(42)

# Model will be saved here (inside the Model/ subfolder)
MODEL_SAVE_PATH = os.path.join("Model", "model_with_hidden_message.h5")

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)
print(f"Model will be saved to: {MODEL_SAVE_PATH}")

TensorFlow version: 2.20.0
NumPy version: 2.2.6
Model will be saved to: Model\model_with_hidden_message.h5


---
<a id='train'></a>
## 4. Step 1 — Build & Train the Neural Network

We train a simple binary classifier on dummy data. The task doesn't really matter for steganography — we just need a trained model with bias values we can overwrite.

**Dataset:** Student graduation predictor  
- Input features: `[grade (0–100), attendance (0–100%), behavior score (1–5)]`  
- Label: `1` (will graduate) if grade > 60 AND attendance > 75% AND behavior ≥ 3, else `0`

**Architecture:**
```
Input (3 features)
    ↓
Dense(16, relu)   ← hidden layer — this is where we hide the message
    ↓
Dense(1, sigmoid) ← output layer
```

The hidden layer has **16 neurons** → **16 bias values** → can hold up to **16 characters**.

> **Note:** You can increase the hidden layer size to hold a longer message.

In [2]:
# --- Generate dummy student data ---

N_SAMPLES = 1000

# Feature 1: grades (0–100)
# Feature 2: attendance percentage (0–100)
# Feature 3: behavior score (integer 1–5)
X = np.random.rand(N_SAMPLES, 3).astype(np.float32)
X[:, 0] *= 100   # scale grades to 0–100
X[:, 1] *= 100   # scale attendance to 0–100%
X[:, 2] = np.random.randint(1, 6, size=N_SAMPLES)  # behavior: 1, 2, 3, 4, or 5

# Label: graduate (1) if grade > 60, attendance > 75, behavior >= 3
y = ((X[:, 0] > 60) & (X[:, 1] > 75) & (X[:, 2] >= 3)).astype(int)

print(f"Dataset shape: X={X.shape}, y={y.shape}")
print(f"Graduation rate in dummy data: {y.mean()*100:.1f}%")

Dataset shape: X=(1000, 3), y=(1000,)
Graduation rate in dummy data: 5.9%


In [3]:
# --- Build the model ---

model = Sequential([
    # Hidden layer — 16 neurons, relu activation
    # This layer has 16 bias values → can hold up to 16 characters
    Dense(16, activation='relu', input_shape=(3,), name='hidden'),

    # Output layer — 1 neuron, sigmoid activation for binary classification
    Dense(1, activation='sigmoid', name='output')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

c:\Users\Asus Tuf Gaming\miniconda3\envs\ai\lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden (Dense)                  │ (None, 16)             │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 81 (324.00 B)

 Trainable params: 81 (324.00 B)

 Non-trainable params: 0 (0.00 B)

In [4]:
# --- Train the model ---
# We do a quick training run. The biases will be overwritten afterward anyway,
# so we don't need a perfect model — just a realistic starting point.

history = model.fit(X, y, epochs=15, batch_size=32, verbose=1)

# Show the bias values before embedding — these are the "learned" values
pre_biases = model.get_layer('hidden').get_weights()[1]
print(f"\nBias values BEFORE embedding (first 16):")
print(pre_biases)

Epoch 1/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.2040 - loss: 8.3949    
Epoch 2/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4350 - loss: 2.6693 
Epoch 3/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8560 - loss: 0.3162 
Epoch 4/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9390 - loss: 0.1898 
Epoch 5/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9380 - loss: 0.1851 
Epoch 6/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9370 - loss: 0.1830 
Epoch 7/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9410 - loss: 0.1816 
Epoch 8/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9410 - loss: 0.1806 
Epoch 9/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9410 - loss: 0.1798 
Epoch 10/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9380 - loss: 0.1790 
Epoch 11/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9400 - loss: 0.1783 
Epoch 12/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accurac

---
<a id='encode'></a>
## 5. Step 2 — Encode the Secret Message into Bias Values

Now for the fun part. 🐱

We convert the secret message to ASCII integers, then **replace** the hidden layer's bias values with those integers.

**Key constraint:** `len(message)` must be ≤ number of neurons in the hidden layer.

| Message length | Required neurons |
|---|---|
| 13 chars | ≥ 13 neurons |
| 16 chars | ≥ 16 neurons |
| 64 chars | ≥ 64 neurons |

The remaining bias values (neurons after the message) are left at their trained values — they don't carry any message, and this helps the model behave slightly more normally.

> **Change the message below to whatever you want to hide!** nyaa~

In [5]:
# --- Define the secret message ---
# Must be <= number of neurons in the hidden layer (currently 16)
SECRET_MESSAGE = "Cats are cute"   # ← change this!

# Convert each character to its ASCII integer
ascii_values = [ord(char) for char in SECRET_MESSAGE]

print(f"Secret message : '{SECRET_MESSAGE}'")
print(f"Message length : {len(SECRET_MESSAGE)} characters")
print(f"ASCII values   : {ascii_values}")

# Validate that the message fits in the hidden layer
hidden_layer    = model.get_layer('hidden')
weights, biases = hidden_layer.get_weights()
n_neurons       = len(biases)

if len(ascii_values) > n_neurons:
    raise ValueError(
        f"Message is too long! '{SECRET_MESSAGE}' has {len(ascii_values)} characters "
        f"but the hidden layer only has {n_neurons} neurons. "
        f"Shorten the message or increase Dense(neurons) above."
    )

print(f"\nHidden layer capacity : {n_neurons} neurons")
print(f"Characters used       : {len(ascii_values)} / {n_neurons}")

Secret message : 'Cats are cute'
Message length : 13 characters
ASCII values   : [67, 97, 116, 115, 32, 97, 114, 101, 32, 99, 117, 116, 101]

Hidden layer capacity : 16 neurons
Characters used       : 13 / 16


In [6]:
# --- Embed the message into bias values ---

# Copy the existing biases so we don't mutate in-place accidentally
new_biases = biases.copy()

# Replace the first len(message) bias slots with ASCII codes
# The rest of the biases stay as their trained values
new_biases[:len(ascii_values)] = np.array(ascii_values, dtype=np.float32)

# Write the modified biases back into the layer
# Note: we keep the original weights (kernel) untouched
hidden_layer.set_weights([weights, new_biases])

# Verify the embedding
verify_biases = hidden_layer.get_weights()[1]
print("Bias values AFTER embedding:")
print(verify_biases)
print(f"\nFirst {len(ascii_values)} biases match ASCII values: ",
      np.array_equal(verify_biases[:len(ascii_values)],
                     np.array(ascii_values, dtype=np.float32)))

Bias values AFTER embedding:
[6.7000000e+01 9.7000000e+01 1.1600000e+02 1.1500000e+02 3.2000000e+01
 9.7000000e+01 1.1400000e+02 1.0100000e+02 3.2000000e+01 9.9000000e+01
 1.1700000e+02 1.1600000e+02 1.0100000e+02 1.3755988e-01 0.0000000e+00
 1.1175484e-01]

First 13 biases match ASCII values:  True


---
<a id='save'></a>
## 6. Step 3 — Save the Model

The model is saved as a `.h5` file (Keras HDF5 format). This format stores:
- The model architecture (layer types, sizes, names)
- All weights and biases — **including our modified biases**
- The optimizer state and training config

The saved file looks like any other Keras model. Nothing on the outside hints that it carries a hidden message. 🐾

In [7]:
# --- Save the model with the embedded message ---

# Make sure the Model/ directory exists
os.makedirs("Model", exist_ok=True)

model.save(MODEL_SAVE_PATH)

file_size_kb = os.path.getsize(MODEL_SAVE_PATH) / 1024
print(f"✅ Model saved to: {MODEL_SAVE_PATH}")
print(f"   File size: {file_size_kb:.1f} KB")
print(f"   Hidden message: '{SECRET_MESSAGE}'")

✅ Model saved to: Model\model_with_hidden_message.h5
   File size: 22.7 KB
   Hidden message: 'Cats are cute'


---
<a id='decode'></a>
## 7. Step 4 — Decode the Message from the Saved Model

Now we pretend we're on the other side — we only have the `.h5` file and want to find the hidden message.

**The decoding strategy:**
1. Load the model from disk (cold load — as if someone else sent us this file)
2. Extract bias values from every `Dense` layer
3. Round each float to the nearest integer
4. Filter to keep only values in the printable ASCII range (32–126 for readable text, or 0–255 broadly)
5. Convert integers → characters → read the message

> This is also how a CTF solver would approach the challenge: *"something weird is in the biases..."*

In [8]:
# --- Load the model fresh from disk ---
# This simulates receiving the .h5 file and not knowing what's inside

loaded_model = tf.keras.models.load_model(MODEL_SAVE_PATH)
print("Model loaded successfully.")
loaded_model.summary()

Model loaded successfully.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden (Dense)                  │ (None, 16)             │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 83 (336.00 B)

 Trainable params: 81 (324.00 B)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [9]:
# --- Extract all bias values from Dense layers ---

all_bias_values = []

for layer in loaded_model.layers:
    # Only process Dense layers (they have both weights and biases)
    if isinstance(layer, Dense):
        layer_weights = layer.get_weights()

        # get_weights() returns [kernel, bias] for a Dense layer
        if len(layer_weights) == 2:
            bias = layer_weights[1]   # index 1 = bias vector
            all_bias_values.extend(bias.flatten().tolist())
            print(f"  Layer '{layer.name}': {len(bias)} bias values → {bias.tolist()}")

print(f"\nTotal bias values collected: {len(all_bias_values)}")

  Layer 'hidden': 16 bias values → [67.0, 97.0, 116.0, 115.0, 32.0, 97.0, 114.0, 101.0, 32.0, 99.0, 117.0, 116.0, 101.0, 0.13755987584590912, 0.0, 0.11175484210252762]
  Layer 'output': 1 bias values → [-0.1051587462425232]

Total bias values collected: 17


In [10]:
# --- Decode the bias values back into text ---

# Round each float32 to the nearest integer
# (ASCII integers stored as float32 round-trip perfectly)
rounded = [round(v) for v in all_bias_values]

# Keep only values in the printable ASCII range (32=space to 126=~)
# Widen to 0–255 if you suspect non-printable characters in the message
ascii_range = [v for v in rounded if 32 <= v <= 126]

# Convert integers to characters
decoded_message = ''.join(chr(v) for v in ascii_range)

print("🔓 Decoded message from bias values:")
print(f"   '{decoded_message}'")
print(f"\nRaw integers (all bias values, rounded): {rounded}")

🔓 Decoded message from bias values:
   'Cats are cute'

Raw integers (all bias values, rounded): [67, 97, 116, 115, 32, 97, 114, 101, 32, 99, 117, 116, 101, 0, 0, 0, 0]


In [11]:
# --- Targeted decode: read exactly N characters from a specific layer ---
# If you know the message was embedded in the hidden layer specifically,
# you can extract it directly without guessing the length.

MESSAGE_LENGTH = len(SECRET_MESSAGE)  # in a real CTF you would probe for this

hidden_biases = loaded_model.get_layer('hidden').get_weights()[1]
extracted_ascii = hidden_biases[:MESSAGE_LENGTH].astype(int)
targeted_message = ''.join(chr(code) for code in extracted_ascii)

print(f"Targeted extraction from 'hidden' layer, first {MESSAGE_LENGTH} biases:")
print(f"  ASCII codes : {extracted_ascii.tolist()}")
print(f"  Message     : '{targeted_message}'")

# Final check
assert targeted_message == SECRET_MESSAGE, "Mismatch! Something went wrong."
print("\n✅ Encode → Save → Load → Decode: perfect round-trip!")

Targeted extraction from 'hidden' layer, first 13 biases:
  ASCII codes : [67, 97, 116, 115, 32, 97, 114, 101, 32, 99, 117, 116, 101]
  Message     : 'Cats are cute'

✅ Encode → Save → Load → Decode: perfect round-trip!


---
<a id='capacity'></a>
## 8. Capacity Analysis

How much can we hide? It depends on the total number of neurons across all Dense layers.

**Formula:**
```
Max message length (chars) = Σ (neurons in each Dense layer)
```

This cell calculates the exact capacity of the current model.

In [12]:
# --- Capacity analysis ---

print("=" * 55)
print(f"{'Layer Name':<20} {'Neurons':>10} {'Capacity':>12}")
print("=" * 55)

total_capacity = 0
for layer in model.layers:
    if isinstance(layer, Dense):
        n = layer.get_weights()[1].shape[0]   # number of bias values = number of neurons
        total_capacity += n
        print(f"{layer.name:<20} {n:>10} {n:>10} chars")

print("=" * 55)
print(f"{'TOTAL':<20} {total_capacity:>10} {total_capacity:>10} chars")
print(f"\nCurrent message uses {len(SECRET_MESSAGE)} / {total_capacity} capacity "
      f"({len(SECRET_MESSAGE)/total_capacity*100:.1f}%)")

# Tip: spread across layers for longer messages
print("\nTip: to hide a longer message, increase Dense(neurons) or add more layers.")

Layer Name              Neurons     Capacity
hidden                       16         16 chars
output                        1          1 chars
TOTAL                        17         17 chars

Current message uses 13 / 17 capacity (76.5%)

Tip: to hide a longer message, increase Dense(neurons) or add more layers.


---
<a id='limitations'></a>
## 9. Limitations & Future Work

### Current Limitations

| Issue | Details |
|---|---|
| **Detectable** | Bias values are replaced with round integers (e.g., 67, 97, 116). A careful inspector checking the bias distribution would notice these don't look like trained values. |
| **Accuracy impact** | Overwriting learned biases changes the model's predictions. Acceptable for CTF challenges; problematic for watermarking a production model you depend on. |
| **No encryption** | The message is stored as raw ASCII. Anyone who knows to check the biases can read it immediately. Adding AES or XOR encryption before embedding would fix this. |
| **Keras `.h5` only** | This implementation targets TensorFlow/Keras `.h5` files. PyTorch and ONNX would need their own adapters. |

### Future Improvements

- **Subtler embedding:** Instead of replacing the full float value, modify only the **least significant bits of the float32 mantissa**. The value changes by < 0.00001, invisible to inspection.
- **Encryption layer:** XOR or AES-encrypt the message before embedding so raw ASCII isn't readable.
- **Comparison study:** Benchmark against classical stego methods (F5, LSB image stego, DeepStego) for detectability and capacity.
- **PyTorch support:** Adapt for `.pt`/`.pth` checkpoint files.
- **Formal paper:** Submit to arXiv or IEEE student conference.

---

### References

- Paper draft: [Embedding Data inside Neural Network Bias Model (Google Docs)](https://docs.google.com/document/d/1FmyEre04vu4t17ZvqIxB2F5ku5cUlZJ6QiYmIGt1yvg/edit?usp=sharing)
- Repository: [github.com/Pandora41/HiddenBiasSteganography](https://github.com/Pandora41/HiddenBiasSteganography)

---

```
  /\_____/\
 (  ^ ω ^  )   ₊˚ "psst... check the biases"
  >  🐾  <
```

*Made with love (and too much coffee) by a cat-themed researcher. nyaa~*